In [0]:
%sql use schema investment_vision; select current_catalog(), current_schema(); 

In [0]:
%sql
select * from investment_vision.valid_transactions-- where Instrument = 'APPL' ---group by Trans_Code

In [0]:
%python
from pyspark.sql.functions import *
trans_code_df = spark.sql(
    """
SELECT
  a.Instrument,
  a.Trans_Code,
  CAST(ROUND(SUM(Transaction_Amount_Clean), 2) AS DECIMAL(18,2)) AS amount_trans_code,
  b.current_market_value,
  b.current_price,
  b.total_qty
FROM investment_vision.valid_transactions a
LEFT JOIN investment_vision.current_portfolio_value b
  ON a.Instrument = b.ticker
GROUP BY a.Instrument,a.Trans_Code,b.current_market_value,b.current_price,b.total_qty
ORDER BY a.Instrument
    """
)

display(trans_code_df)

In [0]:
%python
from pyspark.sql.functions import *
trans_code_pivot1_df = (
    trans_code_df
    .groupBy(
        "Instrument",
        "total_qty",
        "current_price",
        "current_market_value"
    )
    .pivot("Trans_Code")
    .agg(
        first("amount_trans_code")
    )
)
display(trans_code_pivot1_df)

In [0]:
%python
trans_code_pivot1_df.createOrReplaceTempView("trans_code_pivot1_df")

In [0]:
%sql
CREATE OR REPLACE TABLE investment_vision.current_details AS
SELECT
  a.*,
  b.Sector,
  b.Industry_Type,
  CASE 
    WHEN b.Sector LIKE '%ETF%' THEN 'ETF'
    ELSE 'Equity'
  END AS Invest_Catrgory,
  CASE 
    WHEN a.current_market_value IS NULL THEN 'Inactive'
    ELSE 'Active'
  END AS Present_Status,
  current_date() AS Present_Date
FROM 
  trans_code_pivot1_df a
LEFT JOIN 
  investment_vision.raw_invest_types b
ON 
  a.Instrument = b.CUSIP;

In [0]:
%sql
select * from investment_vision.current_details;

In [0]:
%sql
select
  Invest_Catrgory,
  sum(coalesce(Buy,0)) as total_Buy,
  sum(coalesce(CDIV,0)) as total_CDIV,
  sum(coalesce(Sell,0)) as total_Sell,
  sum(coalesce(current_market_value,0)) as total_current_market_value,
  (sum(coalesce(Buy,0)) - sum(coalesce(Sell,0)) - sum(coalesce(CDIV,0))) as actual_inv
from investment_vision.current_details
group by Invest_Catrgory

In [0]:
%python
from pyspark.sql.functions import *
trans_code_month_df = spark.sql(
    """
SELECT
  a.Instrument,
  a.Trans_Code,
  CAST(ROUND(SUM(Transaction_Amount_Clean), 2) AS DECIMAL(18,2)) AS amount_trans_code,
  a.Settle_Month_Year,
  b.current_market_value,
  b.current_price,
  b.total_qty
FROM investment_vision.valid_transactions a
LEFT JOIN investment_vision.current_portfolio_value b
  ON a.Instrument = b.ticker
GROUP BY a.Instrument,a.Trans_Code,b.current_market_value,b.current_price,b.total_qty,a.Settle_Month_Year
ORDER BY a.Instrument
    """
)

display(trans_code_month_df)

In [0]:
from pyspark.sql.functions import regexp_extract

# Extract month abbreviation as string and year as integer
trans_code_month_df = trans_code_month_df.withColumn(
    "inv_month", regexp_extract(col("Settle_Month_Year"), r"^(\w{3})-\d{4}$", 1)
).withColumn(
    "inv_year", regexp_extract(col("Settle_Month_Year"), r"^(\w{3})-(\d{4})$", 2).cast("int")
)

display(trans_code_month_df)

In [0]:
%python
from pyspark.sql.functions import *
trans_code_pivot1_month_df = (
    trans_code_month_df
    .groupBy(
        "Instrument",
        "Settle_Month_Year",
        "inv_month",
        "inv_year"
    )
    .pivot("Trans_Code")
    .agg(
        first("amount_trans_code")
    )
)
display(trans_code_pivot1_month_df)

In [0]:
%python
trans_code_pivot1_month_df.createOrReplaceTempView("trans_code_pivot1_month_df")

In [0]:
%sql
CREATE OR REPLACE TABLE investment_vision.monthly_inv_details AS
SELECT
  a.*,
  b.Sector,
  b.Industry_Type,
  CASE 
    WHEN b.Sector LIKE '%ETF%' THEN 'ETF'
    ELSE 'Equity'
  END AS Invest_Catrgory,
  -- CASE 
  --   WHEN a.current_market_value IS NULL THEN 'Inactive'
  --   ELSE 'Active'
  -- END AS Present_Status,
  current_date() AS Present_Date
FROM 
  trans_code_pivot1_month_df a
LEFT JOIN 
  investment_vision.raw_invest_types b
ON 
  a.Instrument = b.CUSIP;

In [0]:
%sql
select * from investment_vision.monthly_inv_details order by Instrument,Settle_Month_Year

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
CREATE OR REPLACE TABLE investment_vision.silver_xirr_details AS
SELECT
  a.*,
  b.Sector,
  b.Industry_Type,
  CASE 
    WHEN b.Sector LIKE '%ETF%' THEN 'ETF'
    ELSE 'Equity'
  END AS Invest_Catrgory,
  -- CASE 
  --   WHEN a.current_market_value IS NULL THEN 'Inactive'
  --   ELSE 'Active'
  -- END AS Present_Status,
  current_date() AS Present_Date
FROM 
  investment_vision.raw_xirr_details a
LEFT JOIN 
  investment_vision.raw_invest_types b
ON 
  a.Instrument = b.CUSIP;

In [0]:
%sql
select * from investment_vision.silver_xirr_details where Current_Market_Value > 0 and Holding_Period_days > 365

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.